# 아파트 실거래가 예측

In [1]:
# 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/AI/Data Set/아파트 실거래가 예측 AI"

Mounted at /content/drive
/content/drive/MyDrive/AI/Data Set/아파트 실거래가 예측 AI


# **1. 환경 설정 및 데이터 로드**
**불필요한 중복 임포트를 줄이고 설정을 통합합니다.**

# **환경 설정**

In [3]:
# 라이브러리 설치
!pip install optuna

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.6 MB/s eta 0:00:00


## 데이터 로드

In [4]:
# 시각화 설정
sns.set_theme(style="darkgrid")
plt.rc('font', family='NanumBarunGothic') # 코랩 한글깨짐 방지

# 데이터 로드
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# **2. 핵심 전처리 함수화**
**반복되는 아파트명 정제, 브랜드 추출, 라벨 인코딩 등을 하나의 함수로 묶었습니다.**

## 핵심 전처리

In [5]:
def preprocess_data(train, test):
    df_all = [train, test]

    # 1. 아파트명 정제 (괄호 제거)
    regex = "\(.*\)"
    for df in df_all:
        df['apt'] = df['apt'].apply(lambda x: re.sub(regex, '', x).strip())

    # 2. Top 10 브랜드 및 주요 아파트 키워드 추출
    top10 = ['자이', '푸르지오', '더샵', '롯데캐슬', '이편한|e편한|e-편한', '힐스테이트', '아이파크', '래미안', 'sk|SK|에스케이', '데시앙']
    apt_names = ['그레이스', '양지', '쌍용', '현대', '한신', '삼성', '대우', '신동아', '두산', '주공', '우성', '벽산', '동원로얄듀크', '경남', '삼환', '삼익', '대림', '코오롱', '파크리오', '엘지', '성원', '잠실', '동궁리치웰', '동성']
    keywords = top10 + apt_names

    for df in df_all:
        df['top10'] = 0
        for brand in top10:
            df.loc[df['apt'].str.contains(brand), 'top10'] = 1

        # 키워드에 해당하지 않는 아파트는 'others'로 통일
        df['transformed_apt'] = 'others'
        for kw in keywords:
            df.loc[df['apt'].str.contains(kw), 'transformed_apt'] = kw
        df['apt'] = df['transformed_apt']

    # 3. 동 이름 중복 처리 (서울/부산 구분)
    # 실제 데이터 상황에 맞춰 'city' 정보를 결합하여 고유 'dong' 생성
    for df in df_all:
        df['dong'] = df['city'] + "_" + df['dong']

    # 4. 가격 기반 타겟 인코딩 (Apt, Dong)
    # 주의: Test 데이터에 Leakage가 없도록 Train 기준 평균으로 매핑
    for col in ['apt', 'dong']:
        mapper = train.groupby(col)['transaction_real_price'].mean().sort_values().index
        mapping = {val: i for i, val in enumerate(mapper)}
        train[col] = train[col].map(mapping)
        test[col] = test[col].map(mapping).fillna(-1) # Train에 없는 값 처리

    # 5. 수치형 데이터 처리 (로그 변환 및 스케일링)
    train['log_price'] = np.log1p(train['transaction_real_price'])
    for df in df_all:
        df['log_area'] = np.log1p(df['exclusive_use_area'])
        df['year_of_completion'] -= df['year_of_completion'].min()
        # 층수 음수 처리
        df['floor'] = df['floor'] + 4 if 'floor' in df.columns else df['floor']
        # 시티 이진화
        df['city'] = (df['city'] == '서울특별시').astype(int)

    # 6. 불필요한 컬럼 삭제
    drop_cols = ['transaction_id', 'apartment_id', 'jibun', 'transaction_date', 'addr_kr', 'exclusive_use_area', 'transformed_apt']
    train.drop(drop_cols + ['transaction_real_price'], axis=1, inplace=True)
    test.drop(drop_cols, axis=1, inplace=True)

    return train, test

train_clean, test_clean = preprocess_data(train_df.copy(), test_df.copy())

<>:5: SyntaxWarning: invalid escape sequence '\('
<>:5: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_4956/4002808452.py:5: SyntaxWarning: invalid escape sequence '\('
  regex = "\(.*\)"


# **3. 모델 비교 및 검증**
**TimeSeriesSplit을 활용하여 시계열 특성을 반영한 검증을 수행합니다.**

In [6]:
def evaluate_models(X, y):
    tscv = TimeSeriesSplit(n_splits=5)
    models = {
        "Linear": LinearRegression(),
        "Ridge": Ridge(alpha=0.8),
        "RF": RandomForestRegressor(n_estimators=100, max_depth=9, n_jobs=-1),
        "XGB": xgb.XGBRegressor(n_estimators=500, max_depth=9),
        "LGBM": lgb.LGBMRegressor(n_estimators=500, max_depth=9)
    }

    results = {}
    for name, model in models.items():
        rmses = []
        for train_idx, val_idx in tscv.split(X):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            model.fit(X_tr, y_tr)
            pred = model.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, pred))
            rmses.append(rmse)
        results[name] = np.mean(rmses)
        print(f"{name} Mean RMSE: {results[name]:.4f}")
    return results

# 실행
X = train_clean.drop('log_price', axis=1)
y = train_clean['log_price']
# model_scores = evaluate_models(X, y)

# **4. 하이퍼파라미터 튜닝 및 최종 예측**
**성능이 가장 좋았던 LightGBM을 기준으로 최적화합니다.**

In [9]:
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
    }

    # 마지막 20%를 검증셋으로 사용
    cut = int(len(train_clean) * 0.8)
    X_train, y_train = X.iloc[:cut], y.iloc[:cut]
    X_val, y_val = X.iloc[cut:], y.iloc[cut:]

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=50)])
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))

# 튜닝 시작
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

# 최종 모델 학습 및 제출
final_model = lgb.LGBMRegressor(**study.best_params)
final_model.fit(X, y)
final_preds = np.expm1(final_model.predict(test_clean))

# 저장
submission = pd.DataFrame({
    'transaction_id': test_df['transaction_id'],
    'transaction_real_price': final_preds
})
submission.to_csv('submission_final.csv', index=False)

[I 2026-04-01 16:56:35,076] A new study created in memory with name: no-name-9c471c9a-92ac-42e4-8a5a-147c79a7b97b


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[460]	valid_0's rmse: 0.201639


[I 2026-04-01 16:57:02,870] Trial 0 finished with value: 0.2016392003582378 and parameters: {'max_depth': 10, 'learning_rate': 0.09548679152240167, 'n_estimators': 1436, 'min_child_samples': 10}. Best is trial 0 with value: 0.2016392003582378.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[594]	valid_0's rmse: 0.197729


[I 2026-04-01 16:57:39,627] Trial 1 finished with value: 0.1977290420783387 and parameters: {'max_depth': 7, 'learning_rate': 0.09590589225265067, 'n_estimators': 1130, 'min_child_samples': 33}. Best is trial 1 with value: 0.1977290420783387.


...

[I 2026-04-01 17:17:06,150] Trial 17 finished with value: 0.1968031663079787 and parameters: {'max_depth': 8, 'learning_rate': 0.08200893888267817, 'n_estimators': 1704, 'min_child_samples': 48}. Best is trial 17 with value: 0.1968031663079787.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1214]	valid_0's rmse: 0.196747


[I 2026-04-01 17:18:18,839] Trial 18 finished with value: 0.19674713012233344 and parameters: {'max_depth': 9, 'learning_rate': 0.08572832993569031, 'n_estimators': 1274, 'min_child_samples': 46}. Best is trial 18 with value: 0.19674713012233344.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1207]	valid_0's rmse: 0.196802


[I 2026-04-01 17:19:30,018] Trial 19 finished with value: 0.19680245710575664 and parameters: {'max_depth': 9, 'learning_rate': 0.08415162264717277, 'n_estimators': 1720, 'min_child_samples': 50}. Best is trial 18 with value: 0.19674713012233344.
